# Setup

## Imports

In [ ]:
# Native packages
from pathlib import Path
from importlib import reload
import datetime as dt
import numpy as np
import subprocess
import itertools
import difflib

# Other packages
import matplotlib.pyplot as plt
import metpy.calc as mpc
import jinja2
import pandas as pd
from metpy.units import units

# My packages
import ps
import ps.projects.thermals as pst
import carlee_tools as ct
import xarrams as xrr

plt.style.use("cd_slides")

## RAMSIN diffing

In [ ]:
# ---------------------------------------------------------------------------
# Isolate the RAMSIN change for each parameter individually.
#
# Take the first parameter set as the base. For each parameter, find the
# experiment whose parameter set differs from the base in *exactly* that one
# parameter, and diff its RAMSIN (and submit script) against the base. This
# isolates the effect of each parameter on the rendered files so we can confirm
# each one is wired up correctly.
# ---------------------------------------------------------------------------
def diff_files(path_a, path_b):
    return "\n".join(
        difflib.unified_diff(
            path_a.read_text().splitlines(),
            path_b.read_text().splitlines(),
            fromfile=str(path_a),
            tofile=str(path_b),
        )
    )


def find_single_param_variant(parameter_sets, base_pset, param):
    """Return the experiment name whose pset differs from `base_pset` in only `param`."""
    for name, pset in parameter_sets.items():
        differing = [k for k in base_pset if pset[k] != base_pset[k]]
        if differing == [param]:
            return name
    return None


def diff_ramsins_by_parameter(experiment_ramsins):
    # Use the first parameter set as the base. Recover its parameters from the
    # experiment name with ct.utils.to_kv_pairs (the inverse of ct.to_kv_str).
    experiment_parameter_sets = {
        k: ct.kv.to_kv_pairs(k, parse_floats=True)
        for k in experiment_ramsins.keys()
    }
    base_experiment, base_pset = ct.first_item(experiment_parameter_sets)
    print(f"Base experiment: {base_experiment}")
    print(f"Base parameters: {base_pset}\n")

    for param in base_pset:
        match = find_single_param_variant(
            experiment_parameter_sets, base_pset, param
        )
        if match is None:
            print(
                f"=== '{param}': no single-parameter variant available,"
                " skipping ===\n"
            )
            continue
        match_pset = experiment_parameter_sets[match]
        print(
            f"=== Isolating '{param}': {base_pset[param]} ->"
            f" {match_pset[param]} ==="
        )
        print(f"  base:  {base_experiment}")
        print(f"  match: {match}")
        print("RAMSIN diff:")
        print(
            diff_files(
                experiment_ramsins[base_experiment], experiment_ramsins[match]
            )
        )
        # print("Submit script diff:")
        # print(
        #     diff_files(
        #         experiment_submit_scripts[base_experiment],
        #         experiment_submit_scripts[match],
        #     )
        # )
        print("")

## Matching the template winds

In [ ]:
# ---------------------------------------------------------------------------
# Keep the wind profile identical to the template run.
#
# The corrected soundings below change the thermodynamics, which changes p(z).
# Both the template and the corrected soundings are specified on *pressure*
# levels (IPSFLG=0), so reusing the template's US/VS arrays as-is -- or
# regenerating them analytically -- puts the same wind at the same *pressure*,
# not at the same *height*.
#
# RAMS gives each sounding level the height it gets from hydrostatically
# integrating that sounding (`arrsnd`, RAMS_tracer_source/src/init/rhhi.f90);
# that is the height coordinate the model actually works in, so it is what we
# match on.  For the template sounding it differs from the `z` levels the
# sounding was written on by up to ~2.5 km, so matching on the sounding's own
# `z` column would *not* be equivalent.
# ---------------------------------------------------------------------------
RAMS_RGAS = 287.0  # rconstants.f90
RAMS_G = 9.80


def arrsnd_heights(ps_hpa, ts_degc, rts_percent):
    """Heights RAMS assigns to a sounding, following `arrsnd` in rhhi.f90.

    Assumes the RAMSIN flags used throughout this notebook: IPSFLG=0
    (pressure in mb), ITSFLG=0 (temperature in C), IRTSFLG=3 (RH in %).
    RAMS leaves `hs(1)` at zero, so the first sounding level defines z = 0.
    """
    p_pa = np.asarray(ps_hpa, dtype=float) * 100.0
    t_k = np.asarray(ts_degc, dtype=float) + 273.15
    r_sat = (
        mpc.saturation_mixing_ratio(p_pa * units.Pa, t_k * units.K)
        .to("dimensionless")
        .magnitude
    )
    r = r_sat * np.asarray(rts_percent, dtype=float) * 0.01
    t_v = t_k * (1.0 + 0.61 * r)

    hs = np.zeros_like(p_pa)
    for k in range(1, len(p_pa)):
        hs[k] = (
            hs[k - 1]
            - RAMS_RGAS
            * 0.5
            * (t_v[k] + t_v[k - 1])
            * (np.log(p_pa[k]) - np.log(p_pa[k - 1]))
            / RAMS_G
        )
    return hs


def with_template_winds(sounding, template_ramsin_path):
    """Return `sounding` with US/VS replaced by the template run's U(z), V(z)."""
    template = Path(template_ramsin_path).read_text()
    for flag, expected in [("IPSFLG", 0), ("ITSFLG", 0), ("IRTSFLG", 3)]:
        actual = int(xrr.execution._parse_ramsin_field(template, flag))
        assert actual == expected, (
            f"template {flag} is {actual}, not {expected}; arrsnd_heights"
            " assumes the latter"
        )

    template_fields = {
        field: np.asarray(
            xrr.soundings._parse_ramsin_sounding_array(template, field),
            dtype=float,
        )
        for field in ["PS", "TS", "RTS", "US", "VS"]
    }
    template_z = arrsnd_heights(
        template_fields["PS"], template_fields["TS"], template_fields["RTS"]
    )
    sounding_z = arrsnd_heights(sounding["PS"], sounding["TS"], sounding["RTS"])

    sounding = sounding.copy()
    for var in ["US", "VS"]:
        # Clamping at the ends is fine: both profiles start at the ground and
        # the template winds are constant above the shear layer.
        sounding[var] = np.interp(sounding_z, template_z, template_fields[var])
    return sounding

# Validation

In [ ]:
SLURM_QUEUE = "batch_short"

N_NODES = 4

SUBMIT_SLURM_JOBS = False

experiment_subgroup_dir = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/validation"
)

# Only things we're changing besides paths is litefile output (off) and
# the sounding in the validation case
ramsin_template_path = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/RAMSIN.newtracer_blspinup-no_forcing-infinitecoldbubble4_deltax-500_frqlite-2"
)

RAMS_EXECUTABLE_PATH = Path(
    "/autofs/nccsopen-svm1_home/cmdavis4/projects/tracer_paper/RAMS_tracer_source/bin.rams/rams-6.3.04_tracer_source"
)

ramsins_paths = {}

for fixed in [False, True]:

    experiment_name = f"fixed-{fixed}"
    experiment_dir = experiment_subgroup_dir / experiment_name
    experiment_dir.mkdir(exist_ok=True, parents=False)

    # Don't need the lite files
    base_params = {"FRQLITE": "0"}
    if fixed:
        zs = xrr.get_z_levels(deltaz=25, dzrat=1.1, dzmax=250, nnzp=108)
        sounding = xrr.soundings.wk84_sounding(
            U_s=15 * units("m/s"),
            q_v0=13 * units("g/kg"),
            veering=False,
            shear_layer_depth=3000 * units("m"),
            z_levels=zs,
            p_sfc=1013.2500 * units("hPa"),
        )
        # Only the thermodynamics should differ from the template run
        sounding = with_template_winds(sounding, ramsin_template_path)
        sounding = xrr.calculate_sounding_derived_vars(sounding)
        xrr.plot_sounding_diagnostics(sounding)
        params = xrr.with_updated_sounding_fields(
            base_params, sounding=sounding
        )
    else:
        params = base_params

    ramsin_path, submit_script_path = xrr.execution.setup_rams_run(
        run_dir=experiment_dir,
        # run_name=experiment_subgroup,
        ramsin_parameters=params,
        ramsin_template_path=ramsin_template_path,
        render_kwargs=dict(
            account="atm124",
            modules=ps.machines.get_current_host().slurm.modules_for("rams"),
            mpi_launcher=ps.machines.get_current_host().slurm.mpi_launcher,
            n_nodes=N_NODES,
            n_cores=128 * N_NODES,
            wall_time="20:00:00",
            rams_executable_path=RAMS_EXECUTABLE_PATH,
            queue=SLURM_QUEUE,
        ),
        new_return_signature=True,
    )
    ramsins_paths[experiment_name] = ramsin_path

    if SUBMIT_SLURM_JOBS:
        subprocess.run(["sbatch", str(submit_script_path.resolve())])
    print("")

diff_ramsins_by_parameter(ramsins_paths)

# Correct WK with well-mixed BL

In [ ]:
SLURM_QUEUE = "batch_short"

N_NODES = 12

SUBMIT_SLURM_JOBS = False

experiment_subgroup_dir = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/validation"
)

# Only things we're changing besides paths is litefile output (off) and
# the sounding in the validation case
ramsin_template_path = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/RAMSIN.newtracer_blspinup-no_forcing-infinitecoldbubble4_deltax-500_frqlite-2"
)

RAMS_EXECUTABLE_PATH = Path(
    "/autofs/nccsopen-svm1_home/cmdavis4/projects/tracer_paper/RAMS_tracer_source/bin.rams/rams-6.3.04_tracer_source"
)

ramsins_paths = {}


experiment_name = "correct_wk_wellmixed_qv_fixed"
experiment_dir = experiment_subgroup_dir / experiment_name
experiment_dir.mkdir(exist_ok=True, parents=False)

# Don't need the lite files
base_params = {"FRQLITE": "0"}
zs = xrr.get_z_levels(deltaz=25, dzrat=1.1, dzmax=250, nnzp=108)
sounding = xrr.soundings.wk84_sounding(
    U_s=15 * units("m/s"),
    q_v0=13 * units("g/kg"),
    veering=False,
    shear_layer_depth=3000 * units("m"),
    z_levels=zs,
    p_sfc=1000 * units("hPa"),
)
display(sounding.head(20))

ix_900hpa = np.argmin(np.abs(sounding["PS"] - 900))
z_900hpa = sounding["z"].iloc[ix_900hpa] * units("m")


def well_mixed_theta(z, *args, **kwargs):
    base_theta = xrr.soundings._WK_DEFAULT_THETA_CALCULATION(z, *args, **kwargs)
    ix_900hpa_by_z = np.argmin(np.abs(z - z_900hpa))
    theta_900hPa = base_theta[ix_900hpa_by_z]
    base_theta[:ix_900hpa_by_z] = theta_900hPa
    return base_theta


sounding = xrr.soundings.wk84_sounding(
    U_s=15 * units("m/s"),
    q_v0=13 * units("g/kg"),
    veering=False,
    shear_layer_depth=3000 * units("m"),
    z_levels=zs,
    p_sfc=1000 * units("hPa"),
    theta_fn=well_mixed_theta,
)
# Add the well mixed layer

# Only the thermodynamics should differ from the template run
sounding = with_template_winds(sounding, ramsin_template_path)
sounding = xrr.calculate_sounding_derived_vars(sounding)
xrr.plot_sounding_diagnostics(sounding)
display(sounding.head(20))
params = xrr.with_updated_sounding_fields(base_params, sounding=sounding)

ramsin_path, submit_script_path = xrr.execution.setup_rams_run(
    run_dir=experiment_dir,
    # run_name=experiment_subgroup,
    ramsin_parameters=params,
    ramsin_template_path=ramsin_template_path,
    render_kwargs=dict(
        account="atm124",
        modules=ps.machines.get_current_host().slurm.modules_for("rams"),
        mpi_launcher=ps.machines.get_current_host().slurm.mpi_launcher,
        n_nodes=N_NODES,
        n_cores=128 * N_NODES,
        wall_time="20:00:00",
        rams_executable_path=RAMS_EXECUTABLE_PATH,
        queue=SLURM_QUEUE,
    ),
    new_return_signature=True,
)
ramsins_paths[experiment_name] = ramsin_path

if SUBMIT_SLURM_JOBS:
    subprocess.run(["sbatch", str(submit_script_path.resolve())])
print("")

diff_ramsins_by_parameter(ramsins_paths)

In [ ]:
# Sanity check: on the model's own vertical grid, the corrected run's base-state
# U(z) should be indistinguishable from the template run's.
template_fields = {
    field: np.asarray(
        xrr.soundings._parse_ramsin_sounding_array(
            ramsin_template_path.read_text(), field
        ),
        dtype=float,
    )
    for field in ["PS", "TS", "RTS", "US", "VS"]
}
template_z = arrsnd_heights(
    template_fields["PS"], template_fields["TS"], template_fields["RTS"]
)
sounding_z = arrsnd_heights(sounding["PS"], sounding["TS"], sounding["RTS"])
model_z = zs[zs > 0]

fig, axs = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
for ax, var in zip(axs, ["US", "VS"]):
    on_template = np.interp(model_z, template_z, template_fields[var])
    on_corrected = np.interp(model_z, sounding_z, sounding[var].values)
    ax.plot(on_template, model_z / 1000, lw=3, alpha=0.4, label="template run")
    ax.plot(on_corrected, model_z / 1000, ls="--", label="corrected sounding")
    ax.set(xlabel=f"{var[0]} (m/s)", ylim=(0, 20))
    print(
        f"{var}: max |difference| on the model grid ="
        f" {np.abs(on_template - on_corrected).max():.5f} m/s"
    )
axs[0].set_ylabel("z (km)")
axs[0].legend()

# With the original supercell's thermodynamics

In [ ]:
SLURM_QUEUE = "batch_all"

N_NODES = 8

SUBMIT_SLURM_JOBS = False

experiment_subgroup_dir = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/validation"
)

# Only things we're changing besides paths is litefile output (off) and
# the sounding in the validation case
ramsin_template_path = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/RAMSIN.newtracer_blspinup-no_forcing-infinitecoldbubble4_deltax-500_frqlite-2"
)

# The supercell run whose thermodynamics we are adopting.
sc_ramsin_path = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sc_wk/RAMSIN.newtracer_forcing-grant2014_dx-500_frqlite-2_stratwinds"
)

RAMS_EXECUTABLE_PATH = Path(
    "/autofs/nccsopen-svm1_home/cmdavis4/projects/tracer_paper/RAMS_tracer_source/bin.rams/rams-6.3.04_tracer_source"
)

ramsins_paths = {}

experiment_name = "sc_profile"
experiment_dir = experiment_subgroup_dir / experiment_name
experiment_dir.mkdir(exist_ok=True, parents=False)

In [ ]:
# ---------------------------------------------------------------------------
# Thermodynamics: copied verbatim from the supercell RAMSIN.
#
# PS/TS/RTS go across exactly as they are written there, together with their
# unit flags, so this run ingests bit-for-bit the same thermodynamic sounding
# as the supercell -- no interpolation, no unit round-trip.  The flags have to
# travel with the arrays because the supercell writes its sounding in a
# different form from this template's: IPSFLG=1 (heights, with PS(1) the
# surface pressure), ITSFLG=2 (potential temperature), IRTSFLG=2 (mixing
# ratio), against IPSFLG=0/ITSFLG=0/IRTSFLG=3 here.
# ---------------------------------------------------------------------------
sc_ramsin_text = sc_ramsin_path.read_text()
ramsin_params = (
    {
        "FRQLITE": "2",
        "LITE_VARS": xrr.soundings.to_ramsin_values_str(
            ["WC", "THETA", "PI", "RV"]
        ),
        "NLITE_VARS": 4,
    }
    | {
        field: xrr.to_ramsin_values_str(
            xrr.soundings._parse_ramsin_sounding_array(sc_ramsin_text, field)
        )
        for field in ["PS", "TS", "RTS"]
    }
    | {
        field: xrr.execution._parse_ramsin_field(sc_ramsin_text, field)
        for field in ["IPSFLG", "ITSFLG", "IRTSFLG"]
    }
)

# ---------------------------------------------------------------------------
# Winds: this run's own U(z), V(z), unchanged -- but re-expressed on the
# supercell sounding's levels.
#
# US/VS are given one value per sounding level, and we have just replaced the
# sounding's levels with the supercell's, so the template's US/VS arrays cannot
# be carried over as they stand.  Match on the heights RAMS itself assigns to
# each sounding -- the hydrostatically integrated `hs` of `arrsnd`
# (RAMS_tracer_source/src/init/rhhi.f90) -- the same integration
# `arrsnd_heights` above does, and the coordinate the model actually
# interpolates the sounding onto.  `sounding_df_from_ramsin` reproduces it for
# both sounding forms.
# ---------------------------------------------------------------------------
sc_sounding = xrr.sounding_df_from_ramsin(sc_ramsin_path)
template_sounding = xrr.sounding_df_from_ramsin(ramsin_template_path)

for var in ["US", "VS"]:
    # Clamping at the ends is fine: both soundings start at the ground, and
    # above the template's top (~25.6 km) its winds are already constant.
    ramsin_params[var] = xrr.to_ramsin_values_str(
        np.interp(
            sc_sounding["z"].values,
            template_sounding["z"].values,
            template_sounding[var].values,
        )
    )

ramsin_path, submit_script_path = xrr.execution.setup_rams_run(
    run_dir=experiment_dir,
    ramsin_parameters=ramsin_params,
    ramsin_template_path=ramsin_template_path,
    render_kwargs=dict(
        account="atm124",
        modules=ps.machines.get_current_host().slurm.modules_for("rams"),
        mpi_launcher=ps.machines.get_current_host().slurm.mpi_launcher,
        n_nodes=N_NODES,
        n_cores=128 * N_NODES,
        wall_time=(
            "48:00:00" if SLURM_QUEUE != "debug_high_memory" else "02:000:00"
        ),
        rams_executable_path=RAMS_EXECUTABLE_PATH,
        queue=SLURM_QUEUE,
    ),
    new_return_signature=True,
)
ramsins_paths[experiment_name] = ramsin_path

if SUBMIT_SLURM_JOBS:
    subprocess.run(["sbatch", str(submit_script_path.resolve())])

# Nothing outside the sounding block -- the cold-pool forcing, the grid, the
# microphysics -- should have moved, beyond the I/O paths and FRQLITE.
print(diff_files(ramsin_template_path, ramsin_path))

In [ ]:
# ---------------------------------------------------------------------------
# Sanity check the RAMSIN that was actually written.
#
# Re-read its sounding the way RAMS will: the thermodynamics should come back
# identical to the supercell's, and the winds should be indistinguishable from
# the template run's on the model's own vertical grid.
# ---------------------------------------------------------------------------
new_sounding = xrr.sounding_df_from_ramsin(ramsin_path)

for field in ["z", "PS", "TS", "RTS"]:
    max_difference = np.abs(
        new_sounding[field].values - sc_sounding[field].values
    ).max()
    print(f"{field}: max |new - supercell| = {max_difference:.3e}")
    assert max_difference == 0.0, f"{field} does not match the supercell"

model_z = xrr.get_z_levels(deltaz=25, dzrat=1.1, dzmax=250, nnzp=108)
model_z = model_z[model_z > 0]

fig, axs = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
for ax, var in zip(axs, ["US", "VS"]):
    on_template = np.interp(
        model_z, template_sounding["z"], template_sounding[var]
    )
    on_new = np.interp(model_z, new_sounding["z"], new_sounding[var])
    ax.plot(on_template, model_z / 1000, lw=3, alpha=0.4, label="template run")
    ax.plot(on_new, model_z / 1000, ls="--", label="supercell thermodynamics")
    ax.set(xlabel=f"{var[0]} (m/s)", ylim=(0, 20))
    # The residual is not zero here, unlike the corrected-sounding runs above:
    # the supercell sounding is only resolved every 250 m, so its levels cannot
    # quite trace the curvature of the shear layer that the template's much
    # finer near-surface levels do.  It peaks around the top of the shear layer
    # and is well under a percent of the shear.
    print(
        f"{var}: max |template - new| on the model grid ="
        f" {np.abs(on_template - on_new).max():.5f} m/s"
    )
axs[0].set_ylabel("z (km)")
axs[0].legend()

# And the resulting profile itself, for the record.
xrr.plot_sounding_diagnostics(xrr.calculate_sounding_derived_vars(new_sounding))

## History restart 1

The `sc_profile` run stopped two hours in, at 1991-04-26 22:50 -- 6600 s of its
10800 s `TIMMAX` -- when the `debug_high_memory` wall clock ran out. Resume it
from its **latest complete** analysis file. RAMS writes the `-head.txt` only
after closing the companion `.h5` (`anal_write.f90`), so a fully written head
implies a complete data file, but we still open the `.h5` and check that it
exposes every variable in the head manifest, to reject a write that the
wall-clock kill interrupted.

In [ ]:
import xarray as xr

# ================================
# hr1 slurm parameters
# ================================
# The parent ran on debug_high_memory, whose two-hour cap is what stopped it;
# batch_short with the same 12x128 decomposition is what the other validation
# runs here use.  It managed 6600 s of model time in those two hours, and the
# storm is deeper now than it was, so give the remaining 4200 s plenty of room.
HR_SLURM_QUEUE = "batch_all"
HR_N_NODES = 8
HR_WALL_TIME = "48:00:00"
SUBMIT_HISTORY_RESTART_JOBS = False

# ================================
# History-restart parameters
# ================================
# 0 = just finish out the parent's TIMMAX; increase to integrate past it.
restart_extension = dt.timedelta(0)
hr_name = "hr1"
# Copy the HFILIN seed (head + ~5 GB .h5) into the restart directory so the
# restart can't overwrite the file it started from (ICLOBBER=1 in this RAMSIN).
copy_history_seed = True
# Extra RAMSIN overrides for the restart; physics and grid have to match the
# parent, so this stays empty.  setup_history_restart already resets
# IAEROHIST / ITRACHIST / IBUBBLE to 0, which is what we want: the template's
# IBUBBLE=4 and ITRACHIST=1 are initialization-time flags (rdint.f90), so
# leaving them set would re-fire the cold bubble and re-initialize the tracers
# partway through the run instead of carrying both over from the history file.
restart_ramsin_parameters = {}


def latest_valid_head(output_dir, grid=1):
    """Newest ``a-A-*-head.txt`` in *output_dir* whose ``.h5`` opens cleanly and
    contains every variable listed in the head manifest. Returns ``None`` if none
    qualify. Guards the restart against a partially written final analysis file.
    """
    output_dir = Path(output_dir)
    for head in sorted(output_dir.glob("a-A-*-head.txt"), reverse=True):
        data = output_dir / xrr.utils.head_to_data_filename(head, grid=grid)
        if not data.exists():
            continue
        try:
            lines = head.read_text().splitlines()
            n_vars = int(lines[0].strip())
            manifest = {lines[i].split()[0] for i in range(1, n_vars + 1)}
        except (ValueError, IndexError):
            print(f"    {head.name}: truncated/unparseable head, skipping")
            continue
        try:
            with xr.open_dataset(
                data, engine="h5netcdf", phony_dims="access"
            ) as ds:
                present = set(ds.data_vars)
        except Exception as exc:
            print(
                f"    {data.name}: unreadable ({type(exc).__name__}), skipping"
            )
            continue
        missing = manifest - present
        if missing:
            print(
                f"    {data.name}: incomplete ({len(missing)}/{n_vars} vars"
                " missing), skipping"
            )
            continue
        return head
    return None


# Execute history-restart setup
# ================================
experiment_name = "sc_profile"
experiment_dir = experiment_subgroup_dir / experiment_name
# setup_rams_run was called without a run_name above, so the parent RAMSIN is
# {experiment_dir}/RAMSIN -- which is setup_history_restart's default parent.
parent_ramsin_path = experiment_dir / "RAMSIN"

head = latest_valid_head(experiment_dir / "output")
if head is None:
    raise RuntimeError(
        f"No complete analysis output in {experiment_dir / 'output'}"
    )
print(f"HFILIN -> {head.name}")

hr_submit_script_path = xrr.execution.setup_history_restart(
    run_dir=experiment_dir,
    extension=restart_extension,
    hr_name=hr_name,
    hfilin=head,
    parent_ramsin_path=parent_ramsin_path,
    copy_history_seed=copy_history_seed,
    ramsin_parameters=restart_ramsin_parameters,
    render_kwargs=dict(
        account="atm124",
        modules=ps.machines.get_current_host().slurm.modules_for("rams"),
        mpi_launcher=ps.machines.get_current_host().slurm.mpi_launcher,
        n_nodes=HR_N_NODES,
        n_cores=128 * HR_N_NODES,
        wall_time=HR_WALL_TIME,
        rams_executable_path=RAMS_EXECUTABLE_PATH,
        queue=HR_SLURM_QUEUE,
    ),
)
print(hr_submit_script_path)

if SUBMIT_HISTORY_RESTART_JOBS:
    subprocess.run(["sbatch", str(hr_submit_script_path.resolve())])

# The restart RAMSIN should differ from the parent's only in EXPNME, TIMMAX,
# RUNTYPE, HFILIN and the three reset initialization flags -- nothing about the
# sounding, the grid or the microphysics should have moved.
hr_ramsin_path = hr_submit_script_path.parent / f"RAMSIN.{hr_name}"
print(diff_files(parent_ramsin_path, hr_ramsin_path))

## History restart 2

In [ ]:
import xarray as xr

# ================================
# hr1 slurm parameters
# ================================
# The parent ran on debug_high_memory, whose two-hour cap is what stopped it;
# batch_short with the same 12x128 decomposition is what the other validation
# runs here use.  It managed 6600 s of model time in those two hours, and the
# storm is deeper now than it was, so give the remaining 4200 s plenty of room.
HR_SLURM_QUEUE = "batch_high_memory"
HR_N_NODES = 4
HR_WALL_TIME = "24:00:00"
SUBMIT_HISTORY_RESTART_JOBS = False

# ================================
# History-restart parameters
# ================================
# 0 = just finish out the parent's TIMMAX; increase to integrate past it.
restart_extension = dt.timedelta(0)
hr_name = "hr2"
# Copy the HFILIN seed (head + ~5 GB .h5) into the restart directory so the
# restart can't overwrite the file it started from (ICLOBBER=1 in this RAMSIN).
copy_history_seed = True
# Extra RAMSIN overrides for the restart; physics and grid have to match the
# parent, so this stays empty.  setup_history_restart already resets
# IAEROHIST / ITRACHIST / IBUBBLE to 0, which is what we want: the template's
# IBUBBLE=4 and ITRACHIST=1 are initialization-time flags (rdint.f90), so
# leaving them set would re-fire the cold bubble and re-initialize the tracers
# partway through the run instead of carrying both over from the history file.
restart_ramsin_parameters = {}


def latest_valid_head(output_dir, grid=1):
    """Newest ``a-A-*-head.txt`` in *output_dir* whose ``.h5`` opens cleanly and
    contains every variable listed in the head manifest. Returns ``None`` if none
    qualify. Guards the restart against a partially written final analysis file.
    """
    output_dir = Path(output_dir)
    for head in sorted(output_dir.glob("a-A-*-head.txt"), reverse=True):
        data = output_dir / xrr.utils.head_to_data_filename(head, grid=grid)
        if not data.exists():
            continue
        try:
            lines = head.read_text().splitlines()
            n_vars = int(lines[0].strip())
            manifest = {lines[i].split()[0] for i in range(1, n_vars + 1)}
        except (ValueError, IndexError):
            print(f"    {head.name}: truncated/unparseable head, skipping")
            continue
        try:
            with xr.open_dataset(
                data, engine="h5netcdf", phony_dims="access"
            ) as ds:
                present = set(ds.data_vars)
        except Exception as exc:
            print(
                f"    {data.name}: unreadable ({type(exc).__name__}), skipping"
            )
            continue
        missing = manifest - present
        if missing:
            print(
                f"    {data.name}: incomplete ({len(missing)}/{n_vars} vars"
                " missing), skipping"
            )
            continue
        return head
    return None


# Execute history-restart setup
# ================================
experiment_name = "sc_profile"
experiment_dir = experiment_subgroup_dir / experiment_name
# setup_rams_run was called without a run_name above, so the parent RAMSIN is
# {experiment_dir}/RAMSIN -- which is setup_history_restart's default parent.
parent_ramsin_path = experiment_dir / "RAMSIN"

head = latest_valid_head(experiment_dir / "output")
if head is None:
    raise RuntimeError(
        f"No complete analysis output in {experiment_dir / 'output'}"
    )
print(f"HFILIN -> {head.name}")

hr_submit_script_path = xrr.execution.setup_history_restart(
    run_dir=experiment_dir,
    extension=restart_extension,
    hr_name=hr_name,
    hfilin=head,
    parent_ramsin_path=parent_ramsin_path,
    copy_history_seed=copy_history_seed,
    ramsin_parameters=restart_ramsin_parameters,
    render_kwargs=dict(
        account="atm124",
        modules=ps.machines.get_current_host().slurm.modules_for("rams"),
        mpi_launcher=ps.machines.get_current_host().slurm.mpi_launcher,
        n_nodes=HR_N_NODES,
        n_cores=128 * HR_N_NODES,
        wall_time=HR_WALL_TIME,
        rams_executable_path=RAMS_EXECUTABLE_PATH,
        queue=HR_SLURM_QUEUE,
    ),
)
print(hr_submit_script_path)

if SUBMIT_HISTORY_RESTART_JOBS:
    subprocess.run(["sbatch", str(hr_submit_script_path.resolve())])

# The restart RAMSIN should differ from the parent's only in EXPNME, TIMMAX,
# RUNTYPE, HFILIN and the three reset initialization flags -- nothing about the
# sounding, the grid or the microphysics should have moved.
hr_ramsin_path = hr_submit_script_path.parent / f"RAMSIN.{hr_name}"
print(diff_files(parent_ramsin_path, hr_ramsin_path))